# Penn–Fudan pedestrian detection: YOLOv3-tiny baseline
**Stage 2 deadline:** Tuesday, 13 October 2026

Question: how well does a COCO-pretrained YOLOv3-tiny detect pedestrians after fine-tuning on 119 Penn–Fudan images? This notebook fine-tunes the baseline and saves checkpoints, logs, validation predictions and reproducibility details. It uses the prepared dataset without changing the split or resizing the images again.

**Run:** use a fresh Python 3.10–3.13 Jupyter kernel or a Colab GPU runtime. Use the prepared `data/processed` and `data/splits` folders in the repository, or put `pedestrian_data.zip` in the repository root, your home directory, or `/content` in Colab. Run the cells in order. Internet is needed for dependencies, source and pretrained weights. A CUDA GPU is recommended; CPU is supported but slower. The training cell runs **50 epochs**. This notebook ships without claimed training results: results, times and errors are recorded only when executed.

| Requirement | Implementation |
|---|---|
| COCO-pretrained YOLOv3-tiny | Official legacy YOLOv3 model, pinned source commit and release weights |
| One class | `nc: 1`, class 0 = person; incompatible 80-class output layers are reinitialized |
| Training protocol | 640 px, 50 epochs, batch 16, AdamW, initial LR 0.001, seed 42 |
| Data split | Assert 119 train / 25 val / 26 test and sorted names → `random.Random(42).shuffle` |
| Augmentation | Train only; validation has no augmentation or test-time augmentation |
| Deliverables | Best checkpoint, epoch CSV, console logs, configs, environment, runtime, errors, validation predictions |

The 26 test images are checked for data integrity only. No test inference or metrics are run. The model comparison and final test evaluation belong to later stages.

In [ ]:
from pathlib import Path
import sys, os, json, time, hashlib, random, subprocess, zipfile, shutil, traceback
from datetime import datetime, timezone

# Change these only if your files/runtime require it.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / ".git").exists()), Path.cwd()).resolve()
DATA_ROOT = REPO_ROOT  # change this if prepared data lives elsewhere
PROJECT = REPO_ROOT / "baseline" / "runs"
ZIP_PATH = next((p for p in [REPO_ROOT/"pedestrian_data.zip", Path.cwd()/"pedestrian_data.zip", Path.home()/"pedestrian_data.zip",
                           Path("/content/pedestrian_data.zip")] if p.is_file()),
                Path.cwd()/"pedestrian_data.zip")
DEVICE = "auto"  # first CUDA GPU if available, otherwise CPU; can be "cpu" or "0"
EPOCHS, BATCH, IMAGE_SIZE, SEED, LR = 50, 16, 640, 42, 0.001
RUN_NAME = "yolov3_tiny_seed42"  # choose a new name for a separate experiment
COMMIT = "97493de334161fe37c630a4fcf1c83c064526d63"
REPO = PROJECT / "vendor" / "yolov3"
ARTIFACTS = PROJECT / "artifacts" / RUN_NAME
ARTIFACTS.mkdir(parents=True, exist_ok=True)

def save_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf-8")

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

def logged_run(args, label, cwd=None):
    """Stream stdout/stderr, retain failures and wall time even if interrupted."""
    args = list(map(str, args))
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    record = {"command": args, "cwd": str(cwd or Path.cwd()), "started_utc": stamp}
    logfile = ARTIFACTS / f"{label}_{stamp}.log"
    start = time.perf_counter()
    proc = None
    try:
        with logfile.open("w", encoding="utf-8") as f:
            proc = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
            for line in proc.stdout:
                print(line, end=""); f.write(line); f.flush()
            code = proc.wait()
        record["returncode"] = code
        if code: raise subprocess.CalledProcessError(code, args)
        record["status"] = "completed"
    except BaseException as exc:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
        record.update(status="failed_or_interrupted", error=repr(exc), traceback=traceback.format_exc())
        with (ARTIFACTS/"errors.jsonl").open("a") as f: f.write(json.dumps(record)+"\n")
        raise
    finally:
        record.update(wall_seconds=time.perf_counter()-start, log=logfile.name)
        save_json(ARTIFACTS/f"{label}_{stamp}.json", record)
    return record

os.environ.update(WANDB_MODE="disabled", COMET_MODE="DISABLED", MPLBACKEND="Agg",
                  CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONHASHSEED=str(SEED),
                  YOLO_AUTOINSTALL="false")
print("Dataset:", ZIP_PATH, "\nArtifacts:", ARTIFACTS)

## 1. Install and pin the implementation
The source revision and key Ultralytics dependencies are pinned. Other dependencies resolve for the runtime's Python/CUDA versions; the exact installed environment is exported after setup. Use that freeze file on the same platform to reproduce the run. In Colab, keep the supplied matched PyTorch/torchvision installation. If pip upgrades an already imported binary library, restart the kernel and rerun.

Sources: [YOLOv3 repository](https://github.com/ultralytics/yolov3), [pinned training code](https://github.com/ultralytics/yolov3/blob/97493de334161fe37c630a4fcf1c83c064526d63/train.py), [official pretrained release](https://github.com/ultralytics/yolov3/releases/tag/v9.6.0). The upstream code uses the AGPL-3.0 license; preserve its license when redistributing it.

In [ ]:
REPO.parent.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    logged_run(["git", "clone", "https://github.com/ultralytics/yolov3.git", REPO], "clone")
    logged_run(["git", "-C", REPO, "checkout", "--detach", COMMIT], "checkout")
actual_commit = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == COMMIT, "Existing source has another revision; use a fresh PROJECT directory."
assert not subprocess.check_output(["git", "-C", REPO, "status", "--porcelain"], text=True).strip(), "Source must be clean."
constraints = PROJECT/"constraints.txt"
constraints.write_text("ultralytics==8.4.164\nultralytics-thop==2.2.1\n")
logged_run([sys.executable, "-m", "pip", "install", "-r", REPO/"requirements.txt",
            "-c", constraints, "ipython"], "install")

In [ ]:
import platform
import numpy as np
import pandas as pd
import yaml
import torch, torchvision
from PIL import Image, ImageDraw
from IPython.display import display
import matplotlib.pyplot as plt

if DEVICE == "auto": DEVICE = "0" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cpu" or torch.cuda.is_available(), "CUDA is unavailable; select CPU or a GPU runtime."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
environment = {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "cuda_runtime": torch.version.cuda, "device": DEVICE,
    "gpu": torch.cuda.get_device_name(int(DEVICE)) if DEVICE != "cpu" else None,
    "source_commit": actual_commit}
save_json(ARTIFACTS/"environment.json", environment)
(ARTIFACTS/"requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
display(environment)

## 2. Check the prepared data
Use the prepared folders when available. Otherwise, extract the supplied ZIP into the ignored run directory. ZIP extraction is tied to its SHA-256 so changed archives cannot mix with old files.

Checks cover image/label pairs, duplicate content, dimensions, class IDs, box boundaries, statistics and the exact seeded split. Mask-to-box conversion is not repeated here. An image-level split does not guarantee scene independence.

In [ ]:
zip_hash = None
if (DATA_ROOT/"data/processed/images/train").is_dir():
    DATA = DATA_ROOT
    assert (DATA/"data/splits/statistics.json").is_file(), "Prepared data needs the split manifests and statistics."
else:
    assert ZIP_PATH.is_file(), f"Place pedestrian_data.zip at {ZIP_PATH} or edit ZIP_PATH."
    DATA = PROJECT/"prepared"
    zip_hash = sha256(ZIP_PATH)
    marker = DATA/"archive.sha256"
    if DATA.exists():
        assert marker.is_file() and marker.read_text().strip() == zip_hash, "Use a new PROJECT for a different/incomplete extraction."
    else:
        with zipfile.ZipFile(ZIP_PATH) as z:
            for member in z.infolist():
                dest = (DATA/member.filename).resolve()
                assert dest.is_relative_to(DATA.resolve()), f"Unsafe ZIP path: {member.filename}"
                assert (member.external_attr >> 16) & 0o170000 != 0o120000, "Symlinks are not accepted."
            z.extractall(DATA)
        marker.write_text(zip_hash)
PROCESSED = DATA/"data/processed"
SPLITS = DATA/"data/splits"
stats = json.loads((SPLITS/"statistics.json").read_text())
manifest = {s:(SPLITS/f"{s}.txt").read_text().split() for s in ("train", "val", "test")}
assert [len(manifest[s]) for s in manifest] == [119,25,26]
all_names = sum(manifest.values(), [])
assert len(set(all_names)) == 170, "Duplicate names / overlapping splits."
shuffled = sorted(all_names); random.Random(SEED).shuffle(shuffled)
for s, expected in zip(manifest, [shuffled[:119], shuffled[119:144], shuffled[144:]]):
    assert set(manifest[s]) == set(expected), f"Seeded split mismatch: {s}"

images, audit, seen_hashes = {}, {}, {}
def read_labels(path):
    rows = [list(map(float, row.split())) for row in path.read_text().splitlines() if row.strip()]
    a = np.asarray(rows, dtype=float).reshape(-1, 5)
    assert np.isfinite(a).all() and (a[:,0] == 0).all(), f"Invalid class/value: {path}"
    assert ((a[:,1:] >= 0) & (a[:,1:] <= 1)).all(), f"Non-normalized box: {path}"
    assert (a[:,3:] > 0).all(), f"Empty box: {path}"
    assert (a[:,1:3]-a[:,3:5]/2 >= -1e-5).all() and (a[:,1:3]+a[:,3:5]/2 <= 1+1e-5).all(), f"Box outside image: {path}"
    assert len(np.unique(a, axis=0)) == len(a), f"Duplicate box: {path}"
    return a

for split, names in manifest.items():
    files = sorted(p for p in (PROCESSED/"images"/split).iterdir() if p.is_file())
    images[split] = {p.stem:p for p in files}
    assert len(files) == len(names) and set(images[split]) == set(names)
    assert {p.stem for p in (PROCESSED/"labels"/split).glob("*.txt")} == set(names)
    boxes = 0
    for name, path in images[split].items():
        with Image.open(path) as im:
            assert im.size == (640,640), f"Unexpected size: {path}"
            im.load()
            digest = hashlib.sha256(im.convert("RGB").tobytes()).hexdigest()
        assert digest not in seen_hashes, f"Duplicate pixels: {name} and {seen_hashes.get(digest)}"
        seen_hashes[digest] = name
        boxes += len(read_labels(PROCESSED/"labels"/split/f"{name}.txt"))
    audit[split] = {"images":len(names), "instances":boxes}
    assert audit[split]["images"] == stats["splits"][split]["images"]
    assert boxes == stats["splits"][split]["instances"]
assert sum(v["instances"] for v in audit.values()) == stats["total_instances"] == 423
# An absolute runtime path avoids resolving data relative to the upstream repository.
DATA_YAML = PROJECT/"dataset.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path":str(PROCESSED), "train":"images/train",
    "val":"images/val", "test":"images/test", "nc":1, "names":{0:"person"}}, sort_keys=False))
save_json(ARTIFACTS/"dataset_audit.json", {"archive_sha256":zip_hash, "seed":SEED, "splits":audit,
    "raw_masks_rechecked":False, "split_verified":True})
shutil.copytree(SPLITS, ARTIFACTS/"splits", dirs_exist_ok=True)
shutil.copy2(DATA_YAML, ARTIFACTS/"dataset.yaml")
display(pd.DataFrame(audit).T)
save_json(ARTIFACTS/"dataset_files_sha256.json", {
    str(p.relative_to(DATA)):sha256(p)
    for folder in (PROCESSED/"images", PROCESSED/"labels", SPLITS)
    for p in sorted(folder.rglob("*")) if p.is_file() and p.suffix in {".png", ".jpg", ".txt", ".json"}
})

In [ ]:
# Visual inspection: train/validation only. Green = supplied ground-truth boxes.
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (split, name) in zip(axes.flat, [(s,n) for s in ("train","val") for n in sorted(manifest[s])[:3]]):
    im = Image.open(images[split][name]).convert("RGB"); draw = ImageDraw.Draw(im)
    for _, x, y, w, h in read_labels(PROCESSED/"labels"/split/f"{name}.txt"):
        draw.rectangle([(x-w/2)*640,(y-h/2)*640,(x+w/2)*640,(y+h/2)*640], outline="lime", width=2)
    ax.imshow(im); ax.set_title(f"{split}: {name}"); ax.axis("off")
fig.tight_layout(); fig.savefig(ARTIFACTS/"ground_truth_check.png", dpi=140); display(fig); plt.close(fig)

## 3. Explicit training configuration
The requested values are fixed below; no automatic batch-size or optimizer selection is used. Training uses the existing 640×640 letterboxed images.

Implementation details to preserve for YOLO11n comparison:
- AdamW initial LR 0.001, beta1 0.9, beta2 0.999, weight decay 0.0005; linear decay factor 0.01. The learning rate changes during warmup and scheduling.
- The upstream nominal batch size is 64: batch 16 uses up to 4 gradient-accumulation steps after warmup (effective batch about 64). A final partial batch has 7 images.
- Warmup is configured as 3 epochs, but the implementation enforces at least 100 iterations. With 8 batches per epoch this is about **12.5 epochs**. The actual epoch learning rates are saved in `results.csv`.
- HSV, translation, scale, horizontal flips and mosaic use the saved low-augmentation configuration. Extra optional Albumentations transforms are disabled so installed optional packages cannot change the experiment. Mosaic remains active through the final epoch; YOLO11n may have a different default.
- AutoAnchor, EMA and automatic AMP checking are upstream behavior; inspect the training log for their actual outcomes. Validation below explicitly uses FP32. Losses and architectures differ between models.
- Early stopping is disabled to complete 50 epochs. `best.pt` is selected on validation fitness = 0.1 × mAP50 + 0.9 × mAP50–95, not test results.
- Seed 42 is passed directly to single-process training. Same seeds do not guarantee identical results across devices/software versions.

This is a fixed baseline run, not hyperparameter search. Changes following OOM or another error should use a new run name and be reported as deviations.

In [ ]:
hyp = yaml.safe_load((REPO/"data/hyps/hyp.scratch-low.yaml").read_text())
hyp.update(lr0=LR, lrf=0.01, momentum=0.9, weight_decay=0.0005,
           warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.0)
HYP_PATH = ARTIFACTS/"hyp_requested.yaml"
HYP_PATH.write_text(yaml.safe_dump(hyp, sort_keys=False))
TRAIN_DIR = ARTIFACTS/"training"
VAL_DIR = ARTIFACTS/"validation"
WEIGHTS = PROJECT/"pretrained/yolov3-tiny.pt"
WEIGHTS.parent.mkdir(exist_ok=True)
WEIGHTS_URL = "https://github.com/ultralytics/yolov3/releases/download/v9.6.0/yolov3-tiny.pt"
# Download explicitly: no silent fallback to a different model/release or random initialization.
if not WEIGHTS.is_file():
    logged_run([sys.executable, "-c", "import urllib.request,sys; urllib.request.urlretrieve(sys.argv[1],sys.argv[2])",
                WEIGHTS_URL, WEIGHTS.with_suffix(".part")], "download_weights")
    WEIGHTS.with_suffix(".part").replace(WEIGHTS)
assert WEIGHTS.stat().st_size > 1_000_000
save_json(ARTIFACTS/"pretrained.json", {"url":WEIGHTS_URL, "sha256":sha256(WEIGHTS)})

# Isolate legacy top-level imports from the notebook kernel and disable optional external loggers.
# This does not modify upstream source files.
LAUNCHER = PROJECT/"launch_yolov3.py"
LAUNCHER.write_text("""import os, sys, runpy
from pathlib import Path
repo, script = Path(sys.argv[1]).resolve(), sys.argv[2]
sys.argv = [str(repo/script)] + sys.argv[3:]
sys.path.insert(0, str(repo))
for name in ('wandb', 'clearml', 'comet_ml', 'albumentations'):
    sys.modules[name] = None
os.chdir(repo)
runpy.run_path(str(repo/script), run_name='__main__')
""")
command = [sys.executable, LAUNCHER, REPO, "train.py", "--weights", WEIGHTS,
    "--cfg", REPO/"models/yolov3-tiny.yaml", "--data", DATA_YAML, "--hyp", HYP_PATH,
    "--imgsz", IMAGE_SIZE, "--epochs", EPOCHS, "--batch-size", BATCH,
    "--optimizer", "AdamW", "--seed", SEED, "--device", DEVICE, "--workers", 0,
    "--patience", 0, "--project", ARTIFACTS, "--name", "training"]
save_json(ARTIFACTS/"run_config.json", {"model":"YOLOv3-tiny", "pretrained":"COCO",
    "epochs":EPOCHS,"batch_size":BATCH,"imgsz":IMAGE_SIZE,"optimizer":"AdamW","lr0":LR,
    "seed":SEED,"device":DEVICE,"command":list(map(str,command)),"source_commit":COMMIT,
    "accumulation_after_warmup":4,"nominal_batch_size":64,"warmup_min_iterations":100,
    "validation_only":True,"optional_albumentations":False})
print(" ".join(map(str, command)))

## 4. Train the baseline
This cell performs the full run. It refuses to overwrite an existing training directory. On failure, the console log and `errors.jsonl` retain the evidence. Diagnose the error first; use a new `RUN_NAME` and rerun the configuration cells for a fresh attempt. Download/copy artifacts out of Colab before the runtime disconnects.

In [ ]:
assert not TRAIN_DIR.exists(), "Existing training run: inspect it or choose a new RUN_NAME; do not mix results."
training_record = logged_run(command, "train", cwd=REPO)
save_json(ARTIFACTS/"training_runtime.json", training_record)
BEST = TRAIN_DIR/"weights/best.pt"
assert BEST.is_file(), "Training did not produce best.pt."
history = pd.read_csv(TRAIN_DIR/"results.csv")
history.columns = history.columns.str.strip()
assert len(history) == EPOCHS, f"Expected {EPOCHS} epochs, found {len(history)}."
assert np.isfinite(history.select_dtypes(include="number").to_numpy()).all(), "Nonfinite training results."
print(f"Completed {len(history)} epochs in {training_record['wall_seconds']/60:.2f} minutes (whole training process).")
display(history.tail())

## 5. Validate the best checkpoint and export predictions
Validation uses only the 25 validation images, confidence floor 0.001 for AP, NMS IoU 0.6, max 300 detections/image, batch 16 and FP32. Upstream precision/recall are measured at its best smoothed F1 operating point at IoU 0.5; they are **not** precision/recall at confidence 0.25. AP is the pinned repository's evaluator, not a claim of official COCOeval equivalence.

The worker writes native YOLO predictions (`class cx cy w h confidence`, normalized to the supplied image) and a simple JSON schema with pixel `xyxy` boxes, including empty predictions. The same schema can be used for YOLO11n predictions and shared evaluation. Keep preprocessing, confidence floor, NMS, max detections, precision and metric definitions consistent before comparing models. Latencies below are validation-pass timings, not a controlled final hardware benchmark.

In [ ]:
EVALUATOR = PROJECT/"evaluate_yolov3.py"
EVALUATOR.write_text("import sys, json, time\nfrom pathlib import Path\nrepo, data, weights, out, device = sys.argv[1:]\nsys.path.insert(0, repo)\nfor name in ('wandb', 'clearml', 'comet_ml', 'albumentations'):\n    sys.modules[name] = None\nimport torch\nfrom models.experimental import attempt_load\nfrom val import run\nmodel = attempt_load(weights, device=torch.device('cpu'), fuse=False)\nassert model.model[-1].nc == 1, 'Checkpoint must have one class'\nparameter_count = sum(p.numel() for p in model.parameters())\ndel model\nstart = time.perf_counter()\nresults, maps, timings = run(data=data, weights=weights, batch_size=16, imgsz=640,\n    conf_thres=0.001, iou_thres=0.6, max_det=300, task='val', device=device, workers=0,\n    single_cls=False, augment=False, save_txt=True, save_conf=True, save_json=False,\n    project=Path(out).parent, name=Path(out).name, exist_ok=False, half=False, plots=True)\nmetrics = dict(zip(('precision_best_f1','recall_best_f1','mAP50','mAP50_95'), map(float,results[:4])))\nmetrics.update(model='YOLOv3-tiny', split='val', images=25, parameters=parameter_count,\n    confidence_floor=0.001, nms_iou=0.6, max_det=300, inference_precision='FP32', batch_size=16,\n    validation_wall_seconds=time.perf_counter()-start,\n    preprocess_ms_per_image=float(timings[0]), inference_ms_per_image=float(timings[1]),\n    nms_ms_per_image=float(timings[2]), evaluator='pinned ultralytics/yolov3 val.run')\nPath(out,'metrics.json').write_text(json.dumps(metrics,indent=2,allow_nan=False))\n", encoding="utf-8")
BEST = TRAIN_DIR/"weights/best.pt"
assert BEST.is_file(), "Run training first."
assert not VAL_DIR.exists(), "Validation output already exists; use a fresh run/output path."
logged_run([sys.executable, EVALUATOR, REPO, DATA_YAML, BEST, VAL_DIR, DEVICE], "validate", cwd=REPO)
metrics = json.loads((VAL_DIR/"metrics.json").read_text())
assert all(0 <= metrics[k] <= 1 for k in ("precision_best_f1","recall_best_f1","mAP50","mAP50_95"))
pd.DataFrame([metrics]).to_csv(VAL_DIR/"metrics.csv", index=False)
display(pd.DataFrame([metrics]))

predictions = []
for name in sorted(manifest["val"]):
    f = VAL_DIR/"labels"/f"{name}.txt"
    detections = []
    if f.exists():
        for row in f.read_text().splitlines():
            cls, x,y,w,h,conf = map(float,row.split())
            assert cls == 0 and 0 <= conf <= 1
            detections.append({"class_id":0,"score":conf,
                "bbox_xyxy":[(x-w/2)*640,(y-h/2)*640,(x+w/2)*640,(y+h/2)*640]})
    predictions.append({"image_id":name,"file_name":images["val"][name].name,
                        "width":640,"height":640,"detections":detections})
save_json(VAL_DIR/"predictions.json", {"schema":"person_xyxy_pixels_v1", "split":"val",
    "coordinate_space":"prepared 640x640 letterboxed images", "images":predictions})
assert len(predictions) == 25
for f in [VAL_DIR/"PR_curve.png", TRAIN_DIR/"results.png"]:
    if f.exists(): display(Image.open(f))

## 6. Save the outputs
The ZIP below contains the best checkpoint, validation predictions, training diagnostics, requested and resolved settings, environment freeze, dataset manifests and hashes, command logs, runtimes, and generated helper scripts.

Main outputs are `training/weights/best.pt`, `training/results.csv`, `validation/predictions.json`, `validation/labels/`, native validation metrics and plots, and the saved configuration and environment files. The generated `evaluate_yolov3.py` is a model-specific export helper.

Training wall time includes startup and the trainer's final validation; inference timing is saved separately. Check the sample boxes and verify all 50 epoch rows and `best.pt` before reporting baseline results. Save this notebook and copy the output ZIP out of Colab before its runtime disconnects. Keep generated datasets, checkpoints and run folders out of Git.

In [ ]:
required = [TRAIN_DIR/"weights/best.pt", TRAIN_DIR/"results.csv", TRAIN_DIR/"opt.yaml",
    TRAIN_DIR/"hyp.yaml", VAL_DIR/"metrics.json", VAL_DIR/"predictions.json",
    ARTIFACTS/"training_runtime.json", ARTIFACTS/"environment.json"]
assert all(p.is_file() for p in required), "Incomplete handoff: inspect missing outputs."
assert len(pd.read_csv(TRAIN_DIR/"results.csv")) == EPOCHS
for p in [LAUNCHER, EVALUATOR, constraints]: shutil.copy2(p, ARTIFACTS/p.name)
# Verify portable outputs without requiring the original machine's absolute dataset path.
(ARTIFACTS/"dataset_portable.yaml").write_text(yaml.safe_dump({"path":"data/processed",
    "train":"images/train","val":"images/val","test":"images/test","nc":1,"names":{0:"person"}}))
save_json(ARTIFACTS/"artifact_sha256.json", {str(p.relative_to(ARTIFACTS)):sha256(p)
    for p in ARTIFACTS.rglob("*") if p.is_file() and p.name != "artifact_sha256.json"})
archive = shutil.make_archive(str(PROJECT/f"{RUN_NAME}_handoff"), "zip", ARTIFACTS)
print("Handoff:", archive)
print("Best checkpoint:", BEST)
print("Validation metrics:", VAL_DIR/"metrics.csv")
# Colab: download now or copy the ZIP to mounted Drive before ending the session.
# from google.colab import files
# files.download(archive)